In [10]:
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

boundaries_path = "../data/raw/boundaries/geoBoundaries-MOZ-ADM2.geojson"
raster_path = "../data/raw/worldpop/moz_under_age_18_2019/moz_T_Under_18_2019_CN_100m_R2025A_v1.tif"

boundaries = gpd.read_file(boundaries_path)

stats = zonal_stats(
    boundaries,
    raster_path,
    stats=["sum", "count"],
    nodata=-99999,
)

stats_df = pd.DataFrame(stats)

print(stats_df.head())
print(stats_df.shape)
print(stats_df.isna().sum())

"""
====================================================================================
Identify the 2 districts with missing 'sum' values
====================================================================================
"""
print("\n")
results = boundaries.copy()
results["under18_sum"] = stats_df["sum"]
results["valid_cell_count"] = stats_df["count"]

missing = results[results["under18_sum"].isna()]

print(missing[["shapeName", "shapeID", "shapeType", "valid_cell_count", "under18_sum"]])

print(missing["valid_cell_count"].describe())

"""
====================================================================================
See if cell-center rule is missing tiny islands
====================================================================================
"""
print("\n")
stats_all_touched = zonal_stats(
    boundaries,
    raster_path,
    stats=["sum", "count"],
    nodata=-99999,
    all_touched=True,
)

stats_all_touched_df = pd.DataFrame(stats_all_touched)

comparison = boundaries[["shapeName", "shapeID"]].copy()
comparison["count_default"] = stats_df["count"]
comparison["sum_default"] = stats_df["sum"]
comparison["count_all_touched"] = stats_all_touched_df["count"]
comparison["sum_all_touched"] = stats_all_touched_df["sum"]

print(
    comparison.loc[
        comparison["shapeName"].isin(["Ilha Licom", "Ilha Risunodo"]),
        [
            "shapeName",
            "count_default",
            "sum_default",
            "count_all_touched",
            "sum_all_touched",
        ],
    ]
)


    count            sum
0  363972  213755.062500
1   51808   92980.007812
2   81424  204043.000000
3  180952  263586.812500
4   45246  103711.765625
(159, 2)
count    0
sum      2
dtype: int64
        shapeName                  shapeID shapeType  valid_cell_count  \
53     Ilha Licom  85939544B53859921366962      ADM2                 0   
54  Ilha Risunodo  85939544B49635827229265      ADM2                 0   

    under18_sum  
53          NaN  
54          NaN  
count    2.0
mean     0.0
std      0.0
min      0.0
25%      0.0
50%      0.0
75%      0.0
max      0.0
Name: valid_cell_count, dtype: float64


        shapeName  count_default  sum_default  count_all_touched  \
53     Ilha Licom              0          NaN                  0   
54  Ilha Risunodo              0          NaN                  0   

    sum_all_touched  
53              NaN  
54              NaN  
